## Lecture 9: Testing & Documentation

### ****Milestone 1:** Test Suite**

****Goal:** create `test_mandelbrot.py` with a meaningful test suite. **Requirements:****

- **$\ge 3$ individual test functions on core computation functions (not on timing, plotting, or I/O)**

I have chosen the function from slide 20.

- **At least one test using `@pytest.mark.parametrize` — across multiple inputs *or* across multiple implementations**

- **Assertions must verify *correctness*, not just “runs without crashing”**

Noted!

In [1]:
%pycat test_mandelbrot.py

from multiprocessing import Pool
import pytest
import numpy as np
from numba import njit

## Function to test
def mandelbrot_pixel(c: complex, max_iter: int) -> int:
    # From slide 20
    z = 0j
    for n in range(max_iter):
        if z.real*z.real + z.imag*z.imag > 4.0:
            return n
        z = z*z + c
    return max_iter

## >= 3 test functions
# Cases based on the ones from lecture 9 code_examples.md
def test_origin():
    assert mandelbrot_pixel(0+0j, 100) == 100

def test_far_outside():
    # escapes on iteration 1
    assert mandelbrot_pixel(5.0+0j, 100) == 1

def test_left_tip():
    assert mandelbrot_pixel(-2.5+0j, 100) == 1

## Using pytest.mark.parametrize (across multiple inputs)
KNOWN_CASES = [
    (0+0j,    100, 100),   # origin: never escapes
    (5.0+0j,  100,   1),   # far outside, escapes on iteration 1
    (-2.5+0j, 100,   1),   # left tip of set
]

@pytest.mark.parametrize("c, max_iter, expected", KNOWN_CASES)
def test_pixel_all(c, max_iter, expected):
   

**What are trustworthy test values?**

- ****Analytically provable** — derived from the definition, no implementation needed. Choose points whose behaviour you can prove from the mathematics alone and state why.**

**$z_0 = 0+0i$**

$$z_1 = (0^2) + (0^2) = 0 + 0i$$

Then ->:

$z_{n+1} = 0+0i$

Thus it never escapes.

**$z_0 = 5+0i$**

$$z_1 = (5^2) + (0^2) = 25 + 0i$$

Thus: it will escape at the first iteration.

**$z_0 = -2.5 + 0i$**

$$z_1 = (-2.5^2)+(0^2) = 25 + 0i$$

Thus: it also will escape at the first iteration.

- ****Cross-validation** — naive loop as oracle; test that NumPy/Numba/multiprocessing agree on a small grid (32×32)**

Done. Please see the code above.

- ****Float32 note** — cross-validate float32 implementations against each other, not against float64 (boundary pixels will differ)**

Noted! I am not mixing any float32 and float64 implementations

**Run and record:**

1. `pytest -v` — note pass count

In [2]:
!pytest -v

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/conda/envs/nsc/bin/python3.11
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: /ncs
plugins: cov-7.1.0, hypothesis-6.152.1
collected 12 items                                                             

l09_ex1_test.py::test_ex1_step1 PASSED                                   [  8%]
l09_ex1_test.py::test_ex1_step2 PASSED                                   [ 16%]
test_mandelbrot.py::test_origin PASSED                                   [ 25%]
test_mandelbrot.py::test_far_outside PASSED                              [ 33%]
test_mandelbrot.py::test_left_tip PASSED                                 [ 41%]
test_mandelbrot.py::test_pixel_all[0j-100-100] PASSED                    [ 50%]
test_mandelbrot.py::test_pixel_all[(5+0j)-100-1] PASSED                  [ 58%]
test_mandelbrot.py::test_pixel_all[(-2.5+0j)-100-1] PASSED               

The pass count is 12.

2. `pytest --cov=. -v` — note coverage %

In [3]:
!pytest --cov=. -v

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/conda/envs/nsc/bin/python3.11
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: /ncs
plugins: cov-7.1.0, hypothesis-6.152.1
collected 12 items                                                             

l09_ex1_test.py::test_ex1_step1 PASSED                                   [  8%]
l09_ex1_test.py::test_ex1_step2 PASSED                                   [ 16%]
test_mandelbrot.py::test_origin PASSED                                   [ 25%]
test_mandelbrot.py::test_far_outside PASSED                              [ 33%]
test_mandelbrot.py::test_left_tip PASSED                                 [ 41%]
test_mandelbrot.py::test_pixel_all[0j-100-100] PASSED                    [ 50%]
test_mandelbrot.py::test_pixel_all[(5+0j)-100-1] PASSED                  [ 58%]
test_mandelbrot.py::test_pixel_all[(-2.5+0j)-100-1] PASSED               

In our file of interest `test_mandelbrot.py` it is 69% coverage.

****Done?** Record pass count and coverage % in your performance notebook (MP3) → commit.**

```sh
mikkel@samos:~/Sync/AAU/Semester 8/github/nsc-mikkel$ git log -n 1
commit dc8ce60c2cb9b92be67cba7d585d1c4ae059026b (HEAD -> main)
Author: mikkel-coder <mikkel.ks.sorensen@gmail.com>
Date:   Sun Apr 19 13:07:30 2026 +0000

    l09: milestone 1 done
```

### ****Milestone 2:** Docstrings and Type Hints**

```python
def mandelbrot_pixel(c: complex, max_iter: int) -> int:
    """Compute the escape iteration count for one complex point.

    Iterates z_{n+1} = z_n^2 + c from z_0 = 0 until
    |z| > 2 (escape) or reaching max_iter.

    Parameters
    ----------
    c : complex
        Complex coordinate to test for set membership.
    max_iter : int
        Maximum iterations; returned if trajectory does not escape.

    Returns
    -------
    int
        First iteration k where |z_k| > 2, or max_iter.
    """
    ...
```

****Goal:** fully document one version of your Mandelbrot code — docstrings and type hints on every public function; verify with `ruff check`.**

#### ****Step 1:** — pick one implementation and add to every public function:**

- **A docstring in NumPy style (one-line summary + Parameters + Returns)**

- **Type hints on all parameters and return values**

#### ****Step 2:** — run ruff:**

```sh
mamba install ruff
ruff check your_mandelbrot.py # fix all reported issues before committing
```

**Type hints also power Pylance:**

- **In VSCode: Settings → Settings → search “type checking mode” → set to `basic`**

- **Pylance uses your annotations to flag type mismatches in real time (Problems tab)**

****Done?** `ruff check` reports 0 errors → commit.**